In [5]:
import os
import pandas as pd

base_dir = os.getcwd()  # 실행 위치: C:\ai_x\source\Pks_Develop

# 수정된 경로 반영
url_folder = os.path.join(base_dir, "N시기별음식URL수집", "N월별조회메뉴수집_비중반영")
search_file = os.path.join(base_dir, "N시기별음식URL수집", "상세메뉴_보정결과_비중포함_순서유지.xlsx")
output_file = os.path.join(base_dir, "상세메뉴_통합결과_URL포함.xlsx")

# --- URL 기반 데이터 수집 함수 ---
def collect_url_data(folder_path):
    excel_files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx") and "상세메뉴" not in f]
    all_data = pd.DataFrame()
    for file in excel_files:
        try:
            digits = ''.join(filter(str.isdigit, os.path.splitext(file)[0]))
            ym_str = None
            for i in range(len(digits) - 5):
                seg = digits[i:i+6]
                if seg.startswith("20"):
                    ym_str = f"{seg[:4]}-{seg[4:]}"
                    break
            if not ym_str:
                continue
            path = os.path.join(folder_path, file)
            df = pd.read_excel(path, sheet_name="요약", dtype=str)
            url_col = next((col for col in df.columns if '비중반영' in col and 'URL수' in col), None)
            if not url_col or '메뉴' not in df.columns:
                continue
            df = df[['메뉴', url_col]].copy()
            df[url_col] = pd.to_numeric(df[url_col], errors='coerce')
            df['월'] = ym_str
            df = df.rename(columns={url_col: '비중반영_URL수'})
            all_data = pd.concat([all_data, df], ignore_index=True)
        except Exception as e:
            print(f"[오류] {file}: {e}")
    return all_data

# --- 병합 및 저장 ---
def merge_url_to_search_file():
    print("URL 수 데이터 수집 중...")
    url_data = collect_url_data(url_folder)

    print("URL 수 피벗 테이블 생성 중...")
    pivot_df = url_data.pivot_table(index='메뉴', columns='월', values='비중반영_URL수')
    pivot_df.index.name = '상세메뉴'
    pivot_df.reset_index(inplace=True)

    print("상세메뉴 원본 파일 로딩 중...")
    df_origin = pd.read_excel(search_file)

    print("상세메뉴 기준으로 병합 중...")
    df_merged = pd.merge(df_origin, pivot_df, how='left', on='상세메뉴')

    print("파일 저장 중...")
    df_merged.to_excel(output_file, index=False)

    print(f"완료: 병합된 엑셀 파일 저장됨\n→ {output_file}")

# 실행
if __name__ == "__main__":
    merge_url_to_search_file()

URL 수 데이터 수집 중...
URL 수 피벗 테이블 생성 중...
상세메뉴 원본 파일 로딩 중...
상세메뉴 기준으로 병합 중...
파일 저장 중...
완료: 병합된 엑셀 파일 저장됨
→ C:\ai_x\source\Pks_Develop\상세메뉴_통합결과_URL포함.xlsx
